# Setup

In [1]:
"""
Indexing pipeline.

Goal: load the 1,259 chunks, generate BGE embeddings on CPU, push to
Qdrant, run a sample semantic search to confirm everything works.
"""
import json
import logging
from pathlib import Path

from finintel.retrieval.chunker import Chunk
from finintel.retrieval.embeddings import Embedder
from finintel.retrieval.vectorstore import VectorStore

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
%load_ext autoreload
%autoreload 2

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS_PATH = PROJECT_ROOT / "data" / "chunks.jsonl"
print(f"Chunks file: {CHUNKS_PATH.relative_to(PROJECT_ROOT)} ({CHUNKS_PATH.stat().st_size / 1024 / 1024:.2f} MB)")

Chunks file: data\chunks.jsonl (4.85 MB)


# Load chunks from JSONL

In [2]:
chunks: list[Chunk] = []
with CHUNKS_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        d = json.loads(line)
        chunks.append(Chunk(**d))
print(f"Loaded {len(chunks):,} chunks")
print(f"Sample: {chunks[0].chunk_id}, {chunks[0].n_tokens} tokens")

Loaded 1,259 chunks
Sample: AAPL_10-K_0000320193-22-000108_mda_000, 715 tokens


# Initialize the embedder (downloads BGE-base on first run)

In [3]:
embedder = Embedder()
# First run downloads ~440 MB to your HF cache (~/.cache/huggingface/).
# Subsequent runs use the cache. Be patient on first execution.
print(f"Model: {embedder.model_name}")
print(f"Dimension: {embedder.dim}")

11:00:21 [INFO] Loading embedding model: BAAI/bge-base-en-v1.5
11:00:21 [INFO] No device provided, using cpu
11:00:21 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
11:00:21 [WARNING] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
11:00:21 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/modules.json "HTTP/1.1 200 OK"
11:00:21 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/modules.json "HTTP/1.1 200 OK"


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\User\Desktop\portfolio_projects\ML_projects\finintel\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\User\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
11:00:22 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

11:00:22 [INFO] Loading SentenceTransformer model from BAAI/bge-base-en-v1.5.
11:00:22 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
11:00:22 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/config_sentence_transformers.json "HTTP/1.1 200 OK"
11:00:23 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
11:00:23 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/README.md "HTTP/1.1 200 OK"
11:00:23 [INFO] HTTP Request: GET https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/README.md "HTTP/1.1 200 OK"


...

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

11:00:25 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
11:00:25 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
11:00:25 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
11:00:26 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
11:00:26 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
11:00:26 [INFO] HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-base-en-v1.5/a5beb1e3e68b9ab74eb54cfd186867f64f240e1a/tokenizer_config.json "HTTP/1.1 200 OK"
11:00:26 [INFO] HTTP Request: HEAD https://huggingface.co/BAAI/bge-base-en-v1.5/resolve/ma

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

11:00:29 [INFO] HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-base-en-v1.5 "HTTP/1.1 200 OK"
C:\Users\User\Desktop\portfolio_projects\ML_projects\finintel\src\finintel\retrieval\embeddings.py:35: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dim = self.model.get_sentence_embedding_dimension()
11:00:29 [INFO] Model loaded. Vector dimension: 768


Model: BAAI/bge-base-en-v1.5
Dimension: 768


# Generate embeddings (this is the slow step) if already cache exist then we can use that

In [6]:
import numpy as np

EMBEDDINGS_CACHE = PROJECT_ROOT / "data" / "embeddings.npy"

if EMBEDDINGS_CACHE.exists():
    print(f"Loading cached embeddings from {EMBEDDINGS_CACHE.relative_to(PROJECT_ROOT)}")
    embeddings = np.load(EMBEDDINGS_CACHE)
    print(f"Loaded shape: {embeddings.shape}")
    if embeddings.shape[0] != len(chunks):
        print(f"⚠ Cache size mismatch ({embeddings.shape[0]} vs {len(chunks)} chunks). Re-encoding.")
        embeddings = embedder.encode([c.text for c in chunks])
        np.save(EMBEDDINGS_CACHE, embeddings)
else:
    print("No cache found. Encoding from scratch (slow)...")
    embeddings = embedder.encode([c.text for c in chunks])
    np.save(EMBEDDINGS_CACHE, embeddings)
    print(f"Cached {embeddings.shape[0]} vectors to {EMBEDDINGS_CACHE.relative_to(PROJECT_ROOT)}")

print(f"\nEmbeddings: {embeddings.shape}, dtype={embeddings.dtype}")

No cache found. Encoding from scratch (slow)...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Cached 1259 vectors to data\embeddings.npy

Embeddings: (1259, 768), dtype=float32


# Index into Qdrant

In [7]:
store = VectorStore(vector_dim=embedder.dim)
store.reset_collection()  # fresh collection — safe to re-run
n = store.upsert_chunks(chunks, embeddings)
print(f"Upserted {n:,} chunks into '{store.collection}'")

11:21:24 [INFO] HTTP Request: GET http://localhost:6333/collections/finintel_chunks/exists "HTTP/1.1 200 OK"
11:21:24 [INFO] HTTP Request: DELETE http://localhost:6333/collections/finintel_chunks "HTTP/1.1 200 OK"
11:21:24 [INFO] Deleted existing collection: finintel_chunks
11:21:24 [INFO] HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
11:21:25 [INFO] HTTP Request: PUT http://localhost:6333/collections/finintel_chunks "HTTP/1.1 200 OK"
11:21:25 [INFO] Created collection finintel_chunks (dim=768, distance=COSINE)
11:21:25 [INFO] HTTP Request: PUT http://localhost:6333/collections/finintel_chunks/points?wait=true "HTTP/1.1 200 OK"
11:21:25 [INFO] HTTP Request: PUT http://localhost:6333/collections/finintel_chunks/points?wait=true "HTTP/1.1 200 OK"
11:21:25 [INFO] HTTP Request: PUT http://localhost:6333/collections/finintel_chunks/points?wait=true "HTTP/1.1 200 OK"
11:21:25 [INFO] HTTP Request: PUT http://localhost:6333/collections/finintel_chunks/points?wait=true "HTTP/1.1 200

Upserted 1,259 chunks into 'finintel_chunks'


# First semantic search!

In [8]:
query = "How does the company describe risks from artificial intelligence?"
query_vec = embedder.encode([query])[0]

print(f"=== Top 5 chunks across all filings ===")
for hit in store.search(query_vec, limit=5):
    print(f"\n[{hit['score']:.3f}] {hit['ticker']:5} {hit['section']:14} {hit['chunk_id']}")
    print(f"  {hit['text'][:280]}...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

11:21:35 [INFO] HTTP Request: POST http://localhost:6333/collections/finintel_chunks/points/query "HTTP/1.1 200 OK"


=== Top 5 chunks across all filings ===

[0.754] MSFT  risk_factors   MSFT_10-K_0000950170-25-100235_risk_factors_009
  . These risks, if realized, may increase our costs, damage our reputation, or adversely affect our results of operations. 22 PART I Item 1A Issues in the development, deployment, and use of AI may result in reputational or competitive harm or liability . We are building AI into m...

[0.749] TSLA  risk_factors   TSLA_10-K_0001628280-26-003952_risk_factors_000
  ITEM 1A. RISK FACTORS You should carefully consider the risks described below together with the other information set forth in this report, which could materially affect our business, financial condition and future results. The risks described below are not the only risks facing ...

[0.746] MSFT  risk_factors   MSFT_10-K_0000950170-24-087843_risk_factors_009
  . Our products may also collect large amounts of data in manners which may not satisfy customers or regulatory requirements. Our customers also operate 

In [9]:
# Now constrained to one company
print(f"=== Top 3 GOOGL chunks only ===")
for hit in store.search(query_vec, limit=3, ticker="GOOGL"):
    print(f"\n[{hit['score']:.3f}] {hit['section']:14} {hit['chunk_id']}")
    print(f"  {hit['text'][:280]}...")

11:21:35 [INFO] HTTP Request: POST http://localhost:6333/collections/finintel_chunks/points/query "HTTP/1.1 200 OK"


=== Top 3 GOOGL chunks only ===

[0.744] risk_factors   GOOGL_10-K_0001652044-25-000014_risk_factors_008
  . As a result of these and other challenges associated with innovative technologies, our implementation of AI systems could subject us to competitive harm, regulatory action, legal liability (including under new and proposed legislation and regulations), new applications of exist...

[0.741] risk_factors   GOOGL_10-K_0001652044-24-000022_risk_factors_008
  . Unintended consequences, uses, or customization of our AI tools and systems may negatively affect human rights, privacy, employment, or other social concerns, which may result in claims, lawsuits, brand or reputational harm, and increased regulatory scrutiny, any of which could...

[0.737] risk_factors   GOOGL_10-K_0001652044-26-000018_risk_factors_011
  . The development and implementation of AI technologies may further increase our exposure to or exacerbate the risks of cyber attacks or other security incidents, particularly

In [10]:
# And a totally different query, restricted to MD&A
query2 = "What was the impact of foreign exchange rates on operating income?"
qv2 = embedder.encode([query2])[0]

print(f"=== Top 3 MD&A chunks across all companies ===")
for hit in store.search(qv2, limit=3, section="mda"):
    print(f"\n[{hit['score']:.3f}] {hit['ticker']:5} {hit['chunk_id']}")
    print(f"  {hit['text'][:280]}...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

11:21:36 [INFO] HTTP Request: POST http://localhost:6333/collections/finintel_chunks/points/query "HTTP/1.1 200 OK"


=== Top 3 MD&A chunks across all companies ===

[0.735] GOOGL GOOGL_10-K_0001652044-25-000014_mda_008
  . We believe the presentation of results on a constant currency basis in addition to U.S. Generally Accepted Accounting Principles (GAAP) results helps improve the ability to understand our performance, because it excludes the effects of foreign currency volatility that are not i...

[0.728] GOOGL GOOGL_10-K_0001652044-24-000022_mda_008
  . 36. Table of Contents Alphabet Inc. Use of Non-GAAP Constant Currency Information International revenues, which represent a significant portion of our revenues, are generally transacted in multiple currencies and therefore are affected by fluctuations in foreign currency exchan...

[0.698] GOOGL GOOGL_10-K_0001652044-23-000016_mda_009
  . Constant currency revenues are calculated by translating current period revenues using prior year comparable period exchange rates, as well as excluding any hedging effects realized in the current period. 33 Tabl